In [1]:
import pandas as pd
from models import Venda, PedidoCompra, EntradaMercadoria, Fornecedor, ProdutoFilial
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from sqlalchemy.dialects.postgresql import insert
from dotenv import load_dotenv
import os

In [2]:
# Importar as bases

# Base vendas
venda_df = pd.read_excel('base_teste_systock.xlsx', sheet_name='venda')

# Base pedido de compra
pedido_df = pd.read_excel('base_teste_systock.xlsx', sheet_name='pedido_compra')

# Base entrada de mercadoria
entrada_df = pd.read_excel('base_teste_systock.xlsx', sheet_name='entradas_mercadoria')

# Base produtos filial
filial_df = pd.read_excel('base_teste_systock.xlsx', sheet_name='produtos_filial')

# Base de fornecedor
fornecedor_df = pd.read_excel('base_teste_systock.xlsx', sheet_name='fornecedor')

### Tratamento dos dados

#### Base venda

In [3]:
# Padronizar tipo de dados

# Copiar venda_df
venda_df_padronizada = venda_df.copy()

# Alterar tipo de dados inteiros
colunas_inteiras = ['venda_id', 'filial_id', 'item']
for coluna in colunas_inteiras:
    venda_df_padronizada[coluna] = venda_df_padronizada[coluna].astype('Int64')

# Alterar tipo de dados para datetime
venda_df_padronizada['data_emissao'] = pd.to_datetime(venda_df_padronizada['data_emissao'], errors='coerce')

# Retirar espaços duplos, do inicio e do final do texto
colunas_texto = ['horariomov', 'produto_id', 'unidade_medida']

for coluna in colunas_texto:
    venda_df_padronizada[coluna] = venda_df_padronizada[coluna].str.strip()

# Alterar tipo de dados para numeric
colunas_numericas = ['qtde_vendida', 'valor_unitario']

for coluna in colunas_numericas:
    venda_df_padronizada[coluna] = pd.to_numeric(venda_df_padronizada[coluna], errors='coerce')

# Remover duplicidade
venda_df_padronizada = venda_df_padronizada.drop_duplicates()

In [4]:
# Validar dados

venda_df_padronizada['erro'] = ''

# 

venda_df_padronizada.loc[venda_df_padronizada['data_emissao'].isna(), 'erro'] += 'data_emissao ausente; '

#
venda_df_padronizada.loc[venda_df_padronizada['produto_id'].isna(), 'erro'] += 'produto_id ausente; '

#
venda_df_padronizada.loc[venda_df_padronizada['qtde_vendida'].isna() | venda_df_padronizada['qtde_vendida'] < 0, 'erro'] += 'qtde_vendida ausente; '

#
venda_df_padronizada.loc[venda_df_padronizada['valor_unitario'].isna() | venda_df_padronizada['valor_unitario'] < 0, 'erro'] += 'valor_unitario inválido; '

In [5]:
# Separar dados válidos
venda_df_valido = venda_df_padronizada[venda_df_padronizada['erro'] == '']

# Dados para revisar
venda_df_pendente = venda_df_padronizada[venda_df_padronizada['erro'] != '']

# Criar arquivo para revisar
if not venda_df_pendente.empty:
    pasta_revisao = "dados_para_revisar"

    os.makedirs(pasta_revisao, exist_ok=True)

    venda_df_pendente.to_excel(
        os.path.join(pasta_revisao, "vendas_para_revisar.xlsx"),
        index=False,
    )

#### Base pedido_compra

A base tem um problema estrutural, ela contém dois blocos de dados na mesma planilha.
As colunas iniciais (0 à 11) possuem uma tabela com cabeçalhos.
A partir da coluna 12, nas linhas finais, aparece um segundo bloco de registros sem cabeçalhos.
Alguns campos também parecem fora da ordem e/ou precisam de validação.

A abordagem adotada será o tratamento por bloco como uma fonte separada, padronizar ambas e depois uni-los

In [6]:
# Dividir em dois blocos
pedido_b1 = pedido_df.iloc[:, :12].copy()

pedido_b2 = pedido_df.iloc[:, 12:].copy()

In [7]:
# Excluir linhas totalmente vazias do bloco 2
pedido_b2 = pedido_b2.dropna(how='all')

# Após validação, observa-se que a coluna 'Unnamed: 21' não possui no primeiro bloco, sendo necessário sua exclusão
pedido_b2 = pedido_b2.drop(columns='Unnamed: 21')

# Obter nomes das colunas
colunas = pedido_b1.columns

# Nomear colunas a partir do primeiro bloco
pedido_b2.columns = colunas[1:11]

# Inserir colunas pedido_id com valores vazios
pedido_b2.insert(0, 'pedido_id', pd.NA)


In [8]:
# Unir os blocos tratados
pedido_unificado = pd.concat([pedido_b1, pedido_b2], ignore_index=True)

In [9]:
# Padronizar tipo de dados

# Alterar tipo de dados para datetime
colunas_data = ['data_pedido','data_entrega']

for coluna in colunas_data:
    pedido_unificado[coluna] = pd.to_datetime(pedido_unificado[coluna], errors='coerce')

# Alterar tipo de dados para Integer
colunas_inteiras = ['pedido_id', 'item', 'ordem_compra', 'qtde_pedida', 'filial_id', 'qtde_entregue', 'fornecedor_id']

for coluna in colunas_inteiras:
    pedido_unificado[coluna] = pedido_unificado[coluna].astype('Int64')


# Alterar tipo de dado para numerica
pedido_unificado['preco_compra'] = pd.to_numeric(pedido_unificado['preco_compra'], errors='coerce')

# Retirar espaços duplos, do inicio e do final do texto
colunas_texto = ['produto_id', 'descricao_produto']

for coluna in colunas_texto:
    pedido_unificado[coluna] = pedido_unificado[coluna].str.strip()

# Remover duplicidade
pedido_unificado = pedido_unificado.drop_duplicates()

In [10]:
# Validação dos dados

# Criar coluna de 'erro'
pedido_unificado['erro'] = ''

# Verificar ausencia de dados e adicionar mensagem em caso positivo
pedido_unificado.loc[pedido_unificado['produto_id'].isna(), 'erro'] += 'produto_id ausente; '

# Verificar ausencia de dados e adicionar mensagem em caso positivo
pedido_unificado.loc[pedido_unificado['descricao_produto'].isna(), 'erro'] += 'descricao_produto ausente; '

# Verificar ausencia de dados ou valor = 0 e adicionar mensagem em caso positivo
pedido_unificado.loc[pedido_unificado['ordem_compra'].isna() | pedido_unificado['ordem_compra'] == 0, 'erro'] += 'ordem_compra ausente; '

# Verificar ausencia de dados ou valor menor que 0(zero) e adicionar mensagem em caso positivo
pedido_unificado.loc[pedido_unificado['qtde_pedida'].isna() | pedido_unificado['qtde_pedida'] < 0, 'erro'] += 'qtde_pedida inválida; '

# Verificar se a quantidade entregue é maior do que a pedida e adicionar mensagem em caso positivo
pedido_unificado.loc[pedido_unificado['qtde_pedida'] < pedido_unificado['qtde_entregue'], 'erro'] += 'qtde_entregue maior que a pedida; '

# Verificar ausencia de dados ou valor menor que 0(zero) e adicionar mensagem em caso positivo
pedido_unificado.loc[pedido_unificado['preco_compra'].isna() | (pedido_unificado['preco_compra'] < 0), 'erro'] += 'preço inválido; '

# Verificar se a data de entrega é menor do que a data de pedido e adicionar mensagem em caso positivo
pedido_unificado.loc[pedido_unificado['data_entrega'] < pedido_unificado['data_pedido'], 'erro'] += 'data_entrega anterior a data_pedido; '

# Verificar ausencia de dados e adicionar mensagem em caso positivo
pedido_unificado.loc[pedido_unificado['fornecedor_id'].isna(), 'erro'] += 'fornecedor_id ausente; '

In [11]:
# Separar dados validos de pendentes
pedido_valido = pedido_unificado[pedido_unificado['erro'] =='']

# Inserir coluna de qtde_pendente
pedido_valido.insert(10, 'qtde_pendente', pedido_valido['qtde_pedida'] - pedido_valido['qtde_entregue'])

pedido_pendentes = pedido_unificado[pedido_unificado['erro'] != '']

# Criar arquivo para revisar
if not pedido_pendentes.empty:
    pasta_revisao = "dados_para_revisar"

    os.makedirs(pasta_revisao, exist_ok=True)

    pedido_pendentes.to_excel(
        os.path.join(pasta_revisao, "pedidos_para_revisar.xlsx"),
        index=False,
    )

#### Base entrada_mercadoria

In [12]:
# Padronizar dados

# Copiar entrada_df
entrada_df_padronizada = entrada_df.copy()

# Alterar tipo de dado para datetime
entrada_df_padronizada['data_entrada'] = pd.to_datetime(entrada_df_padronizada['data_entrada'], errors='coerce')

# Retirar espaços duplos, do inicio e do final do texto
colunas_texto = ['nro_nfe', 'produto_id','descricao_produto']
for coluna in colunas_texto:
    entrada_df_padronizada[coluna] = entrada_df_padronizada[coluna].str.strip()

# Alterar tipo de dados para Integer
colunas_inteiras = ['item', 'ordem_compra', 'qtde_recebida', 'filial_id']
for coluna in colunas_inteiras:
    entrada_df_padronizada[coluna] = entrada_df_padronizada[coluna].astype('Int64')

# Alterar de tipo de dados para numerica
entrada_df_padronizada['custo_unitario'] = pd.to_numeric(entrada_df_padronizada['custo_unitario'], errors='coerce')

# Remover duplicidade
entrada_df_padronizada = entrada_df_padronizada.drop_duplicates()

In [13]:
# Validar dados

# Criar coluna de erro
entrada_df_padronizada['erro'] = ''

#
entrada_df_padronizada.loc[entrada_df_padronizada['nro_nfe'].isna(), 'erro'] += 'nro_nfe ausente; '

#
entrada_df_padronizada.loc[entrada_df_padronizada['ordem_compra'].isna() | entrada_df_padronizada['ordem_compra'] < 0, 'erro'] += 'ordem_compra inválida; '

#
entrada_df_padronizada.loc[entrada_df_padronizada['qtde_recebida'].isna() | entrada_df_padronizada['qtde_recebida'] < 0, 'erro'] += 'qtde_recebida inválida; '

#
entrada_df_padronizada.loc[entrada_df_padronizada['custo_unitario'].isna() | entrada_df_padronizada['custo_unitario'] < 0, 'erro'] += 'custo_unitario inválida; '

In [14]:
# Dados válidos
entrada_df_valido = entrada_df_padronizada[entrada_df_padronizada['erro'] == '']

# Dados para revisar
entrada_df_pendente = entrada_df_padronizada[entrada_df_padronizada['erro'] != '']

# Criar arquivo para revisar
if not entrada_df_pendente.empty:
    pasta_revisao = "dados_para_revisar"

    os.makedirs(pasta_revisao, exist_ok=True)

    entrada_df_pendente.to_excel(
        os.path.join(pasta_revisao, "entradas_mercadoria_para_revisar.xlsx"),
        index=False,
    )

#### Base produtos_filial

In [15]:
# Padronização dos dados

# Copiar df filial
filial_df_padronizada = filial_df.copy()

# Padronizar nomes das colunas
filial_df_padronizada = filial_df_padronizada.rename(columns={'idproduto': 'produto_id', 'idfornecedor':'fornecedor_id'})

# Retirar caractere
filial_df_padronizada['fornecedor_id'] = filial_df_padronizada['fornecedor_id'].str.replace('F', '')

# Alterar tipo de dados para Integer
colunas_inteiras = ['filial_id', 'estoque', 'fornecedor_id']
for coluna in colunas_inteiras:
    filial_df_padronizada[coluna] = filial_df_padronizada[coluna].astype('Int64')

# Retirar espaços duplos, do inicio e do final do texto
colunas_texto = ['produto_id', 'descricao']

for coluna in colunas_texto:
    filial_df_padronizada[coluna] = filial_df_padronizada[coluna].str.strip()

# Alterar de tipo de dados para numerica
colunas_numericas = ['preco_unitario', 'preco_compra', 'preco_venda']

for coluna in colunas_numericas:
    filial_df_padronizada[coluna] = pd.to_numeric(filial_df_padronizada[coluna], errors='coerce')

# Remover duplicidade
filial_df_padronizada = filial_df_padronizada.drop_duplicates()

filial_df_padronizada.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   filial_id       20 non-null     Int64  
 1   produto_id      20 non-null     str    
 2   descricao       20 non-null     str    
 3   estoque         20 non-null     Int64  
 4   preco_unitario  20 non-null     float64
 5   preco_compra    20 non-null     float64
 6   preco_venda     20 non-null     float64
 7   fornecedor_id   20 non-null     Int64  
dtypes: Int64(3), float64(3), str(2)
memory usage: 1.4 KB


In [16]:
# Validar dados

filial_df_padronizada['erro'] = ''

# Verificar ausência de produto_id
filial_df_padronizada.loc[filial_df_padronizada['produto_id'].isna(), 'erro'] += 'produto_id ausente; '

# Verificar ausência de descricao
filial_df_padronizada.loc[filial_df_padronizada['descricao'].isna(), 'erro'] += 'descricao ausente; '

# Verificar ausência de estoque ou menor que 0 (zero)
filial_df_padronizada.loc[filial_df_padronizada['estoque'].isna() | filial_df_padronizada['estoque'] < 0, 'erro'] += 'estoque inválido; '

# Verificar ausência de preco_unitario ou menor que 0 (zero)
filial_df_padronizada.loc[filial_df_padronizada['preco_unitario'].isna() | filial_df_padronizada['preco_unitario'] < 0, 'erro'] += 'preco_unitario inválido; '

# Verificar ausência de preco_compra ou menor que 0 (zero)
filial_df_padronizada.loc[filial_df_padronizada['preco_compra'].isna() | filial_df_padronizada['preco_compra'] < 0, 'erro'] += 'preco_compra inválido; '

# Verificar ausência de preco_venda ou menor que 0 (zero)
filial_df_padronizada.loc[filial_df_padronizada['preco_venda'].isna() | filial_df_padronizada['preco_venda'] < 0, 'erro'] += 'preco_venda inválido; '

# Verificar ausência de fornecedor
filial_df_padronizada.loc[filial_df_padronizada['fornecedor_id'].isna(), 'erro'] += 'fornecedor_id ausente; '

In [17]:
# Separar dados válidos
filial_df_valido = filial_df_padronizada[filial_df_padronizada['erro'] == '']

# Revisar dados
filial_df_pendente = filial_df_padronizada[filial_df_padronizada['erro'] != '']


# Criar arquivo para revisar
if not filial_df_pendente.empty:
    pasta_revisao = "dados_para_revisar"

    os.makedirs(pasta_revisao, exist_ok=True)

    filial_df_pendente.to_excel(
        os.path.join(pasta_revisao, "produtos_filial_para_revisar.xlsx"),
        index=False,
    )

#### Base fornecedor

In [18]:
# Padronizar tipos de dados

# Copiar fornecedor_df original
fornecedor_df_padronizado = fornecedor_df.copy()

# Padronizar nome coluna
fornecedor_df_padronizado = fornecedor_df_padronizado.rename(columns={'idfornecedor': 'fornecedor_id'})

# Remover caractere e converter tipo de dados
fornecedor_df_padronizado['fornecedor_id'] = fornecedor_df_padronizado['fornecedor_id'].str.replace('F', '').astype(int)

# Retirar espaços duplos, do inicio e do final do texto
fornecedor_df_padronizado['razao_social'] = fornecedor_df_padronizado['razao_social'].str.strip()

# Remover duplicidade
fornecedor_df_padronizado = fornecedor_df_padronizado.drop_duplicates(subset='razao_social')

In [19]:
# Validar dados

fornecedor_df_padronizado['erro'] = ''

# Verificar ausência de dados
fornecedor_df_padronizado.loc[fornecedor_df_padronizado['razao_social'].isna(), 'erro'] += 'razao_social ausente; '

# Separar dados válidos
fornecedor_df_valido = fornecedor_df_padronizado[fornecedor_df_padronizado['erro'] == '']

# Revisar dados
fornecedor_df_pendente = fornecedor_df_padronizado[fornecedor_df_padronizado['erro'] != '']

# Criar arquivo para revisar
if not fornecedor_df_pendente.empty:
    pasta_revisao = "dados_para_revisar"

    os.makedirs(pasta_revisao, exist_ok=True)

    fornecedor_df_pendente.to_excel(
        os.path.join(pasta_revisao, "fornecedor_para_revisar.xlsx"),
        index=False,
    )

###  Verificar a existência dos dados no banco e Importar dados

In [20]:
# Conectar com o banco de dados

# Variável de ambiente
load_dotenv()
DATABASE_URL = os.getenv('DATABASE_URL')

db = create_engine(DATABASE_URL)
Session = sessionmaker(bind=db)
session = Session()

In [21]:
# Inserir dados na tabela fornecedor

# Converter DF em dicionário
dados = fornecedor_df_valido.iloc[:, :-1].to_dict(orient='records')


if dados:
    tabela = Fornecedor.__table__
    
    comando = insert(tabela).values(dados).on_conflict_do_nothing( # (ON CONFLICT DO NOTHING) Evitar duplicidade
        index_elements=[
            'fornecedor_id',
            'razao_social'
            ]
            ).returning(
                Fornecedor.fornecedor_id,
                Fornecedor.razao_social
                )

try:
    resultado = session.execute(comando)
    inseridos = resultado.fetchall() # Validar volume dos dados
    session.commit()
except Exception:
    session.rollback()
    raise
else:
    print(f'{Fornecedor.__tablename__}: foram inseridos {len(inseridos)} registros de {len(dados)} válidos')

fornecedor: foram inseridos 0 registros de 20 válidos


In [22]:
# Inserir dados na tabela produtos_filial

# Converter DF em dicionário
registros = filial_df_valido.iloc[:, :-1].to_dict(orient='records')


if registros:
    tabela = ProdutoFilial.__table__
    
    comando = insert(tabela).values(registros).on_conflict_do_nothing( # (ON CONFLICT DO NOTHING) Evitar duplicidade
        index_elements=[
            'filial_id',
            'produto_id'
            ]
            ).returning(
                ProdutoFilial.filial_id,
                ProdutoFilial.produto_id
                )

try:
    resultado = session.execute(comando)
    inseridos = resultado.fetchall() # Validar volume dos dados
    session.commit()
except Exception:
    session.rollback()
    raise
else:
    print(f'{ProdutoFilial.__tablename__}: foram inseridos {len(inseridos)} registros de {len(dados)} válidos')

produtos_filial: foram inseridos 0 registros de 20 válidos


In [23]:
# Inserir dados na tabela pedido_compra

# Converter DF em dicionário
registros = pedido_valido.iloc[:, :-1].to_dict(orient='records')


if registros:
    tabela = PedidoCompra.__table__
    
    comando = insert(tabela).values(registros).on_conflict_do_nothing( # (ON CONFLICT DO NOTHING) Evitar duplicidade
        index_elements=[
            'pedido_id', 
            'produto_id', 
            'item'
            ]
            ).returning(
                PedidoCompra.pedido_id,
                PedidoCompra.produto_id, 
                PedidoCompra.item
                )

try:
    resultado = session.execute(comando)
    inseridos = resultado.fetchall() # Validar volume dos dados
    session.commit()
except Exception:
    session.rollback()
    raise
else:
    print(f'{PedidoCompra.__tablename__}: foram inseridos {len(inseridos)} registros de {len(dados)} válidos')

pedido_compra: foram inseridos 0 registros de 20 válidos


In [24]:
# Inserir dados na tabela entradas_mercadoria

# Converter DF em dicionário
registros = entrada_df_valido.iloc[:, :-1].to_dict(orient='records')


if registros:
    tabela = EntradaMercadoria.__table__
    
    comando = insert(tabela).values(registros).on_conflict_do_nothing( # (ON CONFLICT DO NOTHING) Evitar duplicidade
        index_elements=[
            'ordem_compra',
            'item',
            'produto_id', 
            'nro_nfe'
            ]
            ).returning(
                EntradaMercadoria.ordem_compra,
                EntradaMercadoria.item,
                EntradaMercadoria.produto_id, 
                EntradaMercadoria.nro_nfe
                )

try:
    resultado = session.execute(comando)
    inseridos = resultado.fetchall() # Validar volume dos dados
    session.commit()
except Exception:
    session.rollback()
    raise
else:
    print(f'{EntradaMercadoria.__tablename__}: foram inseridos {len(inseridos)} registros de {len(dados)} válidos')

entradas_mercadoria: foram inseridos 0 registros de 20 válidos


In [25]:
venda_df_valido.iloc[:, :-1]

,venda_id,data_emissao,horariomov,produto_id,qtde_vendida,valor_unitario,filial_id,item,unidade_medida
0,1,2025-01-11,08:00:00,P1,5.00,78.93,1,1,UN
1,2,2025-03-02,08:00:00,P2,7.00,92.96,1,1,UN
2,3,2025-01-28,08:00:00,P3,9.00,197.61,1,1,UN
3,4,2025-01-10,08:00:00,P4,38.60,139.71,1,1,UN
4,5,2025-01-11,08:00:00,P5,3.00,126.79,1,1,UN
5,6,2025-01-24,08:00:00,P6,2.00,36.83,1,1,UN
6,7,2025-02-22,08:00:00,P7,5.00,40.75,1,1,UN
7,8,2025-01-26,08:00:00,P8,20.04,51.37,1,1,UN
8,9,2025-01-17,08:00:00,P9,6.00,172.55,1,1,UN
9,10,2025-01-03,08:00:00,P10,90.00,44.22,1,1,UN


In [26]:
# Inserir dados na tabela venda

# Converter DF em dicionário
dados = venda_df_valido.iloc[:, :-1].to_dict(orient='records')


# print(dados[0].keys())
# print(Venda.__table__.columns.keys())

if dados:
    tabela = Venda.__table__
    
    comando = insert(tabela).values(dados).on_conflict_do_nothing( # (ON CONFLICT DO NOTHING) Evitar duplicidade
        index_elements=[
            'filial_id', 
            'venda_id', 
            'data_emissao', 
            'produto_id', 
            'item', 
            'horariomov'
            ]
            ).returning(
                Venda.filial_id,
                Venda.venda_id,
                Venda.data_emissao,
                Venda.produto_id,
                Venda.item,
                Venda.horariomov
                )

try:
    resultado = session.execute(comando)
    inseridos = resultado.fetchall() # Validar volume dos dados
    session.commit()
except Exception:
    session.rollback()
    raise

else:
    print(f'{Venda.__tablename__}: foram inseridos {len(inseridos)} registros de {len(dados)} válidos')

venda: foram inseridos 33 registros de 33 válidos
